# 14 · Melting the chocolate 🍫☕

We model a **Stefan problem** — a PDE with a *moving boundary* between solid and liquid:
* a **phase field** $\varphi$ (0 = solid, 1 = molten) carries the front in a thin diffuse layer;
* it couples to the temperature $T$ through the **latent heat** of melting:
  warming drives the transition, but turning solid into liquid *swallows* energy and holds the
  front back.

We meet it **first in 1-D** (cheap, high resolution, easy plots), then a **2-D cross-section of a
Toblerone**, and finally the **whole bar in 3-D** — one model, the very same solver, three geometries.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "anywidget", "matplotlib"], check=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from netgen.occ import *
from ngsolve import *
from ngsolve.meshes import Make1DMesh
from ngsolve.webgui import Draw

In [ ]:
def progress(i, n):                                    # tiny dependency-free bar
    import os
    if os.environ.get("WEBGUI_SCENE_DIR"):             # static-site build: stay silent
        return
    if (i + 1) % max(1, n // 100) == 0 or i + 1 == n:
        b = int(28 * (i + 1) / n)
        sys.stdout.write(f"\r  melting… [{'█'*b}{'·'*(28-b)}] {100*(i+1)//n:3d}%")
        sys.stdout.flush()
        if i + 1 == n:
            sys.stdout.write("\n")

## 1. A simple model — a tilted double well

The **phase** relaxes towards solid or liquid through a double-well reaction, across a thin
interface of width $\sim\epsilon$:
$$ \tau\,\partial_t\varphi \;=\; \epsilon^2\,\Delta\varphi
   \;+\; \underbrace{\varphi(1-\varphi)}_{\text{double well}}\bigl(\varphi-\tfrac12+m(T)\bigr),
   \qquad m(T)=\tfrac1\pi\arctan\!\bigl(\gamma\,(T-T_m)\bigr). $$
The corresponding energy is 

$$
\int_{\Omega} \frac{\epsilon^2}{2} \Vert \nabla \varphi \Vert^2 + 
\frac{1}{4}\varphi^2(1-\varphi)^2
-
m(T)\left(
\frac{\varphi^2}{2}
-
\frac{\varphi^3}{3}
\right) dx.
$$
- $\varphi = 0$ and $\varphi = 1$ are local stable equilibria
- the **tilt** $m(T)$ decides which well wins: warm ($T>T_m$) tips it toward liquid, cold toward solid.

The temperature obeys the heat equation with a **latent** sink — each bit of melting draws heat:
$$ \partial_t T \;=\; \alpha\,\Delta T \;-\; L\,\partial_t\varphi . $$

In [ ]:
Tm, tau, alphaT, Lheat, gamma = 0.0, 1e-3, 1.0, 0.6, 4.0     # melt point, relaxation, heat-diff, latent, tilt sharpness


def reaction(phi, T):                                        # R(φ,T) = double well · tilt
    return phi * (1 - phi) * (phi - 0.5 + (1/pi)*atan(gamma*(T - Tm)))

**The tild**: As $T$ rises past $T_m$ the tilt slides the stable well from solid ($\varphi=0$) to liquid ($\varphi=1$).

In [ ]:
# plotting of reaction term and the corresponding energy contribution ...
import numpy as np
import matplotlib.pyplot as plt

phs = np.linspace(0, 1, 200)
Tvals = [-1.0, -0.3, 0.0, 0.3, 1.0]

fig, (ax_reaction, ax_energy) = plt.subplots(
    1, 2,
    figsize=(12.4, 3.2),
    sharex=True
)

for Tval in Tvals:
    mm = (1 / np.pi) * np.arctan(gamma * (Tval - Tm))

    # Reaction term R(phi,T) = -dW/dphi  (local array, *not* the model `reaction`)
    react_vals = phs * (1 - phs) * (phs - 0.5 + mm)

    # Local potential contribution W(phi,T)
    energy = (
        0.25 * phs**2 * (1 - phs)**2
        - mm * (0.5 * phs**2 - phs**3 / 3)
    )

    ax_reaction.plot(
        phs,
        react_vals,
        label=rf"$T={Tval:+.1f}$"
    )

    ax_energy.plot(
        phs,
        energy,
        label=rf"$T={Tval:+.1f}$"
    )

# Left: reaction
ax_reaction.axhline(0, color="k", lw=0.6)
ax_reaction.set_xlabel(r"$\varphi$")
ax_reaction.set_ylabel(r"$R(\varphi,T)$")
ax_reaction.set_title(
    "reaction term — warm melts, cold freezes"
)

# Right: local energy
ax_energy.axhline(0, color="k", lw=0.6)
ax_energy.set_xlabel(r"$\varphi$")
ax_energy.set_ylabel(r"$W(\varphi,T)$")
ax_energy.set_title(
    "local energy — a tilted double well"
)

# One shared legend
handles, labels = ax_reaction.get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="lower center",
    ncol=len(Tvals),
    fontsize=8,
    frameon=False
)

fig.tight_layout(rect=(0, 0.14, 1, 1))
plt.show()

## 2. Discretising it once — and reusing it

Again: 
* diffusion implicit
* reaction explicit
* The hot plate / cold base are **Dirichlet**: we invert only on the **free** dofs

In [ ]:
def run_melt(mesh, gfphi, gfT, dt, eps2, nsteps, nframes):
    """March the phase-field Stefan problem; return the φ- and T-animations as multidim GridFunctions.

    Every snapshot of *both* fields is appended as a multidim component — read out once by `Draw`
    (2-D) or in a small post-processing loop (1-D profiles).
    """
    fes = gfphi.space
    phi, psi = fes.TnT()

    ##setup forms:
    react = BilinearForm(fes, nonassemble=True)                # explicit reaction R(φ,T), applied each step
    react += reaction(phi, gfT) * psi * dx

    M = BilinearForm(phi*psi*dx).Assemble()
    K = BilinearForm(grad(phi)*grad(psi)*dx).Assemble()

    Aphi = M.mat.CreateMatrix(); Aphi.AsVector().data = tau*M.mat.AsVector() + dt*eps2*K.mat.AsVector()
    invphi = Aphi.Inverse(fes.FreeDofs(), inverse="sparsecholesky")     # free dofs only → Dirichlet kept
    AT = M.mat.CreateMatrix(); AT.AsVector().data = M.mat.AsVector() + dt*alphaT*K.mat.AsVector()
    invT = AT.Inverse(fes.FreeDofs(), inverse="sparsecholesky")

    res, phi_old = gfphi.vec.CreateVector(), gfphi.vec.CreateVector()
    animphi = GridFunction(fes, multidim=0); animphi.AddMultiDimComponent(gfphi.vec)
    animT   = GridFunction(fes, multidim=0); animT.AddMultiDimComponent(gfT.vec)

    snap = max(1, nsteps // nframes)
    with TaskManager():
        for step in range(nsteps):
            phi_old.data = gfphi.vec
            react.Apply(gfphi.vec, res)                                  # explicit reaction
            gfphi.vec.data += invphi*(tau*M.mat*gfphi.vec + dt*res - Aphi*gfphi.vec)   # + implicit ε²-diffusion
            dphi = gfphi.vec - phi_old
            gfT.vec.data += invT*(M.mat*(gfT.vec - Lheat*dphi) - AT*gfT.vec)           # latent-heat sink
            if (step + 1) % snap == 0:
                animphi.AddMultiDimComponent(gfphi.vec)
                animT.AddMultiDimComponent(gfT.vec)
            progress(step, nsteps)
    return animphi, animT

## 3. First in 1-D — a heated bar

A bar $[0,4]$ with the **hot plate** at the left ($\varphi=1$, $T=3$); everything else starts
**cold and solid** ($T=-1$, $\varphi=0$). High order is cheap in 1-D, so we use **quartic
elements** and **sample each element at many points** — the plotted profiles are the genuine
higher-order solution (curved *within* every element), not a vertex-to-vertex polyline, as the
melt front marches in.

In [ ]:
mesh1d = Make1DMesh(60, mapping=lambda t: 4.0*t)            # the bar, left end = "left" = hot plate
fes1d  = H1(mesh1d, order=4, dirichlet="left")              # higher order: each element is a quartic
hot = 0                                                     # the left vertex (x=0) is dof 0 — the hot plate
gfphi  = GridFunction(fes1d); gfphi.Set(0.0); gfphi.vec[hot] = 1.0   # solid bar, molten at the hot plate
gfT    = GridFunction(fes1d); gfT.Set(-1.0);  gfT.vec[hot] = 3.0     # cold bar, hot plate at T=3

animphi1d, animT1d = run_melt(mesh1d, gfphi, gfT, dt=1e-3, eps2=4e-4, nsteps=3000, nframes=4)

In [ ]:
# 1-D profiles plot from multidim — ... 
# Read each multidim frame back out of the
# animations and sample it densely (600 points → many *per* quartic element), so the plotted
# curves are the genuine higher-order solution, not a vertex-to-vertex polyline.
xs  = np.linspace(0, 4, 600); pts = [mesh1d(x) for x in xs]
tmp = GridFunction(fes1d)
def profiles(anim):                                          # multidim GridFunction → list of dense 1-D arrays
    out = []
    for vec in anim.vecs:
        tmp.vec.data = vec
        out.append(np.array([tmp(p) for p in pts]))
    return out
phis, Ts = profiles(animphi1d), profiles(animT1d)

elem_x = np.unique([v.point[0] for v in mesh1d.vertices])    # element boundaries (the dofs live in between)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
for a in (a1, a2):                                           # faint ticks: each interval is one quartic element
    for xb in elem_x:
        a.axvline(xb, color="0.92", lw=0.5, zorder=0)
for k, (ph, T) in enumerate(zip(phis, Ts)):
    c = plt.cm.copper(0.2 + 0.8*k/(len(phis)-1))
    a1.plot(xs, ph, color=c, label=f"frame {k}")
    a2.plot(xs, T, color=c)
a1.axhline(0.5, color="gray", lw=0.6, ls="--"); a1.set_title("phase φ (0.5 = the front)")
a2.axhline(Tm, color="gray", lw=0.6, ls="--"); a2.set_title("temperature T ($T_m$ dashed)")
for a in (a1, a2): a.set_xlabel("x")
a1.legend(fontsize=8, frameon=False); fig.tight_layout(); plt.show()

The front advances quickly at first, then **slows** — it has to wait for heat to diffuse deeper,
and every bit of melting *spends* latent heat ($L\,\partial_t\varphi$), which is exactly what holds
a real melt front back.

## 4. The Toblerone — a 2-D cross-section

Now the iconic shape: a **cross-section through a Toblerone**, the row of triangular teeth, resting
on a **cold plate** (the bottom edge). The tooth surfaces are **warm** ($T=2$, molten skin), the
**base** stays cold ($T=-1$, solid) — so the chocolate melts from the surface inward while the base
anchors it. *(The full 3-D bar, clipped through the middle, is the natural next step — left for a
rainy day.)*

In [ ]:
# 2D toblerone-like geometry ...
N, p, H, vh = 3, 1.6, 1.8, 0.4                              # 3 teeth, tooth pitch, peak height, valley height
mout = 0.2                                                  # cut this far *outside* the outer peaks (small → close)
peakx = lambda i: (i + 0.5)*p                               # x of the i-th peak
slope = (H - vh)/(p/2)                                      # the (symmetric) tooth-flank slope
xL, xR = peakx(0) - mout, peakx(N - 1) + mout              # the two vertical cuts, just outside the outer teeth
hcut = H - slope*mout                                       # flank height where the outer cut meets it
# Trace the outline counter-clockwise (so OCC keeps the *inside*): bottom, right cut, teeth, left cut.
wp = WorkPlane().MoveTo(xL, 0).LineTo(xR, 0)                # the bottom (the cold plate)
wp.LineTo(xR, hcut)                                        # right vertical cut, just outside the last tooth
for i in range(N - 1, -1, -1):                             # across the teeth right → left
    wp.LineTo(peakx(i), H)                                  # peak of tooth i
    if i:
        wp.LineTo(i*p, vh)                                  # inner valley to its left (height vh)
wp.LineTo(xL, hcut)                                        # left flank down to the left vertical cut
tob = wp.Close().Face()                                    # Close() draws the left vertical cut back to (xL, 0)
tob.edges.name = "warm"                                    # tooth surfaces + the two cut faces: warm
tob.edges.Min(Y).name = "base"                             # bottom edge: the cold plate
mesh = Mesh(OCCGeometry(tob, dim=2).GenerateMesh(maxh=0.16))
Draw(mesh)

In [ ]:
fes  = H1(mesh, order=3, dirichlet="warm|base")
warm = mesh.Boundaries("warm")
gfphi = GridFunction(fes); gfphi.Set(0.0); gfphi.Set(1.0, definedon=warm)    # molten skin, solid interior + base
gfT   = GridFunction(fes); gfT.Set(-1.0);  gfT.Set(2.0, definedon=warm)      # warm skin, cold interior + base

anim, _ = run_melt(mesh, gfphi, gfT, dt=5e-4, eps2=2e-3, nsteps=200, nframes=16)   # only φ is drawn here

In [ ]:
Draw(anim, mesh, "melting Toblerone — φ (0 = solid, 1 = molten)",
     interpolate_multidim=True, animate=True, min=0, max=1, autoscale=False)

## 5. The whole bar — melting in 3-D

The natural last step: the **real 3-D Toblerone** of [unit 2](02-geometry.ipynb) — three peaks
built as intersecting roofs, the ends cut flat, valley edges filleted. *Nothing* about the solver
changes: the very same `run_melt` runs on a 3-D `H1` space. The whole chocolate skin is **warm**,
only the bottom is the **cold base** — so we **clip the bar through the middle** to watch the melt
front eat inward (on the surface φ is pinned to 1, the story is all on the inside).

:::{dropdown} 🍫 Rebuilding the unit-2 Toblerone (folded — see unit 2 for the walkthrough)
The geometry below is lifted verbatim from unit 2: two intersecting trapezoidal prisms make one
`peak`, peaks are unioned, a `Box` cuts the ends and a flat base, and `MakeFillet` rounds the
valleys.
:::

In [ ]:
# the toblerone mini geometry ...
def trapezoid(a, b, c, d):                                  # a planar quad face from 4 points (unit 2)
    return Face(Wire([Segment(a, b), Segment(b, c), Segment(c, d), Segment(d, a)]))

def peak(cx, cy, z0, bx, by, h, s=0.15):                    # two intersecting roofs → one tooth (unit 2)
    tr_xz = trapezoid(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx+bx/2, cy-by/2, z0),
                      Pnt(cx+s*bx, cy-by/2, z0+h), Pnt(cx-s*bx, cy-by/2, z0+h))
    tr_yz = trapezoid(Pnt(cx-bx/2, cy-by/2, z0), Pnt(cx-bx/2, cy+by/2, z0),
                      Pnt(cx-bx/2, cy+s*by, z0+h), Pnt(cx-bx/2, cy-s*by, z0+h))
    return Prism(tr_xz, Vec(0, by, 0)) * Prism(tr_yz, Vec(bx, 0, 0))

b, depth, Hpk, n_peaks = 2.0, 3.0, 2.2, 3
peaki = lambda i: peak((0.85*i + 0.5)*b, depth/2, 0, b, depth, Hpk, s=0.15)
bar  = sum([peaki(i) for i in range(1, n_peaks)], start=peaki(0))   # union the peaks
Wbar = (0.85*n_peaks + 0.15)*b
bar *= Box(Pnt(0.4*b, 0, 0.1*Hpk), Pnt(Wbar - 0.4*b, depth, Hpk))   # cut the ends flat + a flat base
bar  = bar.MakeFillet([e for e in bar.edges if 0.2*Hpk < e.center[2] < 0.5*Hpk], 0.15)   # round the valleys

bar.faces.name = "warm"                                    # all of the chocolate skin: warm
bar.faces.Min(Z).name = "base"                             # the flat bottom: the cold plate
mesh3 = Mesh(OCCGeometry(bar).GenerateMesh(maxh=0.25))
print(f"the 3-D bar: {mesh3.ne} tetrahedra")
Draw(mesh3)

In [ ]:
fes3  = H1(mesh3, order=3, dirichlet="warm|base")
warm  = mesh3.Boundaries("warm")
gfphi = GridFunction(fes3); gfphi.Set(0.0); gfphi.Set(1.0, definedon=warm)   # molten skin, solid core + base
gfT   = GridFunction(fes3); gfT.Set(-1.0);  gfT.Set(2.0, definedon=warm)     # warm skin, cold core + base

anim, _ = run_melt(mesh3, gfphi, gfT, dt=5e-4, eps2=2e-3, nsteps=200, nframes=10)

In [ ]:
clip = {"x": 0, "y": 1, "z": 0, "dist": 0}

Draw(anim, mesh3, "anim",
     interpolate_multidim=True, animate=True, min=0, max=1, autoscale=False,
     clipping=clip, settings={"Objects": {"Clipping Plane": True}}, euler_angles=[-60, 0, -25])

Clipped open, the 3-D front behaves like the 1-D bar and the 2-D cross-section: it races
in from the warm skin, then **stalls** as it spends latent heat and waits on diffusion — the same
physics, now on the whole bar. 

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):
    _prev = ("13-thermal-plume-hdg", "13 · A puffing thermal plume — HDiv-HDG & HDG 🔥🌀")
    _next = ("15-outlook-unfitted", "15 · Deforming domains (unfitted FEM) 🫧")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))